# Analysis Main Notebook

이 노트북은 `Analysis` 안의 분석 도구를 한 곳에서 호출합니다. SSH 환경에서도 결과를 볼 수 있도록 그래프는 `trends`, JSON과 최종 표 CSV는 `final` 아래에 저장합니다.

## 0. 공통 설정

In [ ]:
from pathlib import Path
import json
import pandas as pd
import sys

# Determine repository root by climbing from the current working directory.
# This makes imports and relative result/output paths work from the notebook
# even if the notebook server starts from a subdirectory.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from Analysis.analyzer import Analyzer
from Analysis.distance_metrics import nearestStats
from Analysis.statistics import calcStats, printStats, reportCluster
from Analysis.result_io import finalPoints, listBands, loadAlgoRuns
from Analysis.trends import (
    plotConverge,
    saveReport,
    coverSummary,
)

RESULTS_ROOT = REPO_ROOT / "__RESULTS__"
ALGORITHM = "drl"          # ga, pso, drl, greedy
MAP_NAME = "gangjin.down" # gangjin.down, seocho.up, ...
SEED_BAND = "40-60"      # None이면 전체 band, 단일 run 분석에는 하나의 band 사용
GRID_M = 5.0
COVERAGE = 45
TARGET_VALUES = (2, 3)

# 구조화된 출력 디렉토리 설정: analysis/{algorithm}/{map_name}/
ANALYSIS_ROOT = RESULTS_ROOT / "analysis"
ALGO_MAP_DIR = ANALYSIS_ROOT / ALGORITHM / MAP_NAME.replace("/", "_")
TRENDS_DIR = ALGO_MAP_DIR / "trends"
FINAL_DIR = ALGO_MAP_DIR / "final"
REPORT_DIR = TRENDS_DIR

# 디렉토리 생성
TRENDS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

RUN_DIR = RESULTS_ROOT / ALGORITHM / MAP_NAME / SEED_BAND if SEED_BAND else RESULTS_ROOT / ALGORITHM / MAP_NAME
RUN_DIR


PosixPath('/workspace/__RESULTS__/pso/gangjin.down/40-60')

## 1. 결과 파일과 seed band 확인

In [18]:
bands = listBands(results_root=RESULTS_ROOT, algorithm=ALGORITHM, map_name=MAP_NAME)
records = loadAlgoRuns(
    results_root=RESULTS_ROOT,
    algorithm=ALGORITHM,
    map_name=MAP_NAME,
    seed_band=SEED_BAND,
)
print("seed bands:", bands)
print("run count in selected band:", len(records))
records[0][0]

seed bands: ['40-60', '60-80', '80-100', '100-120', '120-140']
run count in selected band: 20


PosixPath('/workspace/__RESULTS__/pso/gangjin.down/40-60/20260608_134405.json')

## 2. 단일 run 세대별 추이

`Analyzer`는 하나의 JSON run에 대해 센서 수, 커버리지, fitness 추이를 확인합니다.

In [19]:
analyzer = Analyzer(result_root_path=str(RUN_DIR), file_path=0)

safe_map = MAP_NAME.replace("/", "_").replace(".", "_")

analyzer.plot_evolution_trend(
    xtick_step=50,
    include_corners=True,
    figsize=(10, 3),
    dpi=150,
    show=False,
    save_path=TRENDS_DIR / f"{safe_map}_sensor_count.png",
    close=True,
)
analyzer.plot_coverage_trend(
    xtick_step=50,
    figsize=(10, 3),
    dpi=150,
    show=False,
    save_path=TRENDS_DIR / f"{safe_map}_coverage.png",
    close=True,
)
analyzer.plot_fitness_trend(
    xtick_step=50,
    figsize=(10, 3),
    dpi=150,
    show=False,
    save_path=TRENDS_DIR / f"{safe_map}_fitness.png",
    close=True,
)
print("saved single-run plots to", TRENDS_DIR)


saved single-run plots to /workspace/__RESULTS__/analysis/pso/gangjin.down/trends


## 3. 선택 band의 기본 통계

In [20]:
printStats(str(RUN_DIR))
stats_tuple = calcStats(str(RUN_DIR))
stats_tuple

total runs: 20
[gen=100] coverage mean ± std: 99.8637 ± 0.1356
[final] corner sensors mean ± std: 4.00 ± 0.00
[final] total sensors mean ± std: 23.10 ± 2.30
[time] corner mean ± std (sec): 0.000 ± 0.000
[time] optimizer mean ± std (sec): 99.325 ± 20.339


(20,
 99.86368062317428,
 0.1356220864380137,
 4.0,
 0.0,
 23.1,
 2.300000000000001,
 0.0,
 0.0,
 99.32483635000001,
 20.338752750415203)

## 4. 센서 간 거리 분석

In [21]:
reportCluster(str(RUN_DIR), MAP_NAME, grid_m=GRID_M)

distance_rows = []
for path, run in records:
    d = nearestStats(finalPoints(run), grid_m=GRID_M)
    d["run_name"] = run.get("run_name", path.stem)
    distance_rows.append(d)
distance_df = pd.DataFrame(distance_rows).set_index("run_name")
distance_df.round(3).head()

[gangjin.down] total runs: 20
[final] 평균 군집거리 mean ± std: 47.078 ± 3.083 m
[final] 군집거리 min / max: 34.564 / 49.765 m


,mean_m,std_m,min_m,max_m,n_points
run_name,,,,,
20260608_134405,49.765,5.321,45.000,65.192,21
20260608_134535,48.308,6.205,45.000,74.330,22
20260608_134712,34.564,14.502,7.071,47.170,32
20260608_134942,46.780,2.721,45.000,55.902,22
20260608_135106,47.596,4.000,45.277,61.033,22


## 5. 초기 seed 센서수별 수렴 분석

`metric='best'`는 세대별 best solution 센서수를 기준으로 수렴을 봅니다. `metric='avg'`로 바꾸면 로그의 평균 센서 수 기준으로 분석합니다.

In [22]:
safe_map = MAP_NAME.replace("/", "_").replace(".", "_")
convergence = plotConverge(
    results_root=RESULTS_ROOT,
    algorithm=ALGORITHM,
    map_name=MAP_NAME,
    seed_bands=bands,
    include_corners=True,
    metric="best",
    threshold=0.5,
    dpi=300,
    show=False,
    save_path=TRENDS_DIR / f"convergence_{safe_map}.png",
)
print("convergence generation:", convergence["convergence"]["convergence_gen"])
convergence["convergence"]


convergence generation: 371


{'convergence_gen': 371,
 'by_band': {'40-60': {'gen_from': 2,
   'max_change': 0.44999999999999574,
   'abs_diffs': [0.14999999999999858,
    0.4000000000000057,
    0.25,
    0.09999999999999432,
    0.3500000000000014,
    0.25,
    0.0,
    0.3999999999999986,
    0.3500000000000014,
    0.3500000000000014,
    0.20000000000000284,
    0.29999999999999716,
    0.29999999999999716,
    0.3500000000000014,
    0.20000000000000284,
    0.10000000000000142,
    0.44999999999999574,
    0.14999999999999858,
    0.30000000000000426,
    0.25,
    0.10000000000000142,
    0.25,
    0.09999999999999432,
    0.25,
    0.0,
    0.3500000000000014,
    0.25,
    0.20000000000000284,
    0.29999999999999716,
    0.05000000000000426,
    0.19999999999999574,
    0.20000000000000284,
    0.3999999999999986,
    0.20000000000000284,
    0.04999999999999716,
    0.04999999999999716,
    0.25,
    0.20000000000000284,
    0.20000000000000284,
    0.19999999999999574,
    0.25,
    0.200000000000002

## 6. 전체 커버리지와 중첩 커버리지 분석

In [23]:
report = saveReport(
    results_root=RESULTS_ROOT,
    algorithm=ALGORITHM,
    map_name=MAP_NAME,
    output_dir=REPORT_DIR,
    summary_dir=FINAL_DIR,
    seed_bands=bands,
    include_corners=True,
    metric="best",
    threshold=0.5,
    target_values=TARGET_VALUES,
    dpi=300,
    show=False,
)
print(json.dumps({k: report[k] for k in ["convergence_plot", "coverage_overlap_plot", "coverage_overlap_summary"]}, indent=2))


TypeError: saveReport() got an unexpected keyword argument 'summary_dir'

## 7. 논문 표용 요약 CSV

In [ ]:
summary = coverSummary(
    results_root=RESULTS_ROOT,
    algorithm=ALGORITHM,
    map_name=MAP_NAME,
    seed_bands=bands,
    target_values=TARGET_VALUES,
)
summary_df = pd.DataFrame.from_dict(summary, orient="index")
safe_map = MAP_NAME.replace("/", "_").replace(".", "_")
csv_path = FINAL_DIR / f"summary_{safe_map}.csv"
summary_df.to_csv(csv_path, encoding="utf-8-sig")
print("saved", csv_path)
summary_df[[
    "runs",
    "n_sensors_mean",
    "coverage_percent_mean",
    "overlap_percent_of_target_mean",
    "overlap_percent_of_covered_mean",
    "redundant_hit_percent_mean",
    "logged_final_coverage_mean",
]].round(3)


saved /workspace/__RESULTS__/analysis/ga/gangjin.down/final/summary_gangjin_down.csv


,runs,n_sensors_mean,coverage_percent_mean,overlap_percent_of_target_mean,overlap_percent_of_covered_mean,redundant_hit_percent_mean,logged_final_coverage_mean
40-60,10.0,15.9,98.608,36.095,36.588,28.144,98.608
60-80,10.0,15.7,98.403,35.156,35.717,27.196,98.403
80-100,10.0,16.0,98.695,36.203,36.665,28.143,98.695
100-120,10.0,15.7,98.481,35.015,35.541,27.104,98.481
120-140,10.0,15.6,98.520,37.025,37.552,28.011,98.520
